# **INSTALLING THE REQUIRED LIBRARIES**

In [1]:
import numpy as np
import pandas as  pd
from sklearn.model_selection import train_test_split
from dateutil.relativedelta import relativedelta
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline
from pprint import pprint
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LinearRegression
from sklearn.svm import LinearSVR, SVR
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.metrics import classification_report, accuracy_score, roc_auc_score
import pickle
import os

# **LOADING THE DATASET IN THE CONSOLE **

In [2]:
data = pd.read_csv("/content/train (1).csv")
data = data.dropna()
data.head()

,Employee ID,Date of Joining,Gender,Company Type,WFH Setup Available,Designation,Resource Allocation,Mental Fatigue Score,Burn Rate
0,fffe32003000360033003200,2008-09-30,Female,Service,No,2.0,3.0,3.8,0.16
1,fffe3700360033003500,2008-11-30,Male,Service,Yes,1.0,2.0,5.0,0.36
3,fffe32003400380032003900,2008-11-03,Male,Service,Yes,1.0,1.0,2.6,0.20
4,fffe31003900340031003600,2008-07-24,Female,Service,No,3.0,7.0,6.9,0.52
5,fffe3300350037003500,2008-11-26,Male,Product,Yes,2.0,4.0,3.6,0.29


In [30]:
import os
os.listdir("/content")

['.config', 'sample_data']

In [3]:
from sklearn.model_selection import train_test_split

# remove columns that make dataset huge
X = data.drop(["Burn Rate","Employee ID","Date of Joining"], axis=1)
y = data["Burn Rate"]

# convert text to numbers
X = pd.get_dummies(X, drop_first=True)

# split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Split done")

Split done


In [34]:
# remove rows that contain missing values
data = data.dropna()

In [4]:
from sklearn.linear_model import LinearRegression

linear_regression_model = LinearRegression()
linear_regression_model.fit(X_train, y_train)

print("Model trained")

Model trained


In [5]:
from sklearn.metrics import accuracy_score

# predictions
y_pred = linear_regression_model.predict(X_test)

# convert regression output into burnout class
y_pred = (y_pred > 0.5).astype(int)
y_test_bin = (y_test > 0.5).astype(int)

print("Accuracy:", accuracy_score(y_test_bin, y_pred))

Accuracy: 0.9265734265734266


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
data

In [ ]:
data.head()

In [ ]:
data.tail()

In [ ]:
data["Company Type"].unique()

In [ ]:
data.isnull().sum()

In [ ]:
data.describe()

In [ ]:
data.shape

In [ ]:
data.size

In [ ]:
data.columns.tolist()

In [ ]:
data.nunique()

In [ ]:
data.info()

In [ ]:
data.isnull().sum()

In [ ]:
data.isnull().sum().values.sum()

### **EXPLORATORY DATA ANALYSIS**

In [ ]:
data.corr(numeric_only=True)['Burn Rate'][:-1]


In [ ]:
sns.pairplot(data)
plt.show()

In [ ]:
data = data.dropna()

In [ ]:
data.shape

In [ ]:
data.dtypes

In [ ]:
data_obj = data.select_dtypes(object)
# prints a dictionary of max 10 unique values for each non-numeric column
pprint({ c : data_obj[c].unique()[:10] for c in data_obj.columns})


In [ ]:
data = data.drop('Employee ID', axis = 1)


In [ ]:
data.head()

In [ ]:
print(f"Min date {data['Date of Joining'].min()}")
print(f"Max date {data['Date of Joining'].max()}")
data_month = data.copy()

data_month["Date of Joining"] = data_month['Date of Joining']
data_month["Date of Joining"].groupby(
    data_month['Date of Joining'].dt.month
).count().plot(kind="bar", xlabel='Month', ylabel="Hired employees")


In [ ]:
data_2008 = pd.to_datetime(["2008-01-01"]*len(data))
data["Days"] = data['Date of Joining'].sub(data_2008).dt.days
data.Days

In [ ]:
data.corr(numeric_only=True)['Burn Rate'][:]

In [ ]:
data = data.drop(['Date of Joining','Days'], axis = 1)

In [ ]:
data.head()

In [ ]:
cat_columns = data.select_dtypes(object).columns
fig, ax = plt.subplots(nrows=1, ncols=len(cat_columns), sharey=True, figsize=(10, 5))
for i, c in enumerate(cat_columns):
    sns.countplot(x=c, data=data, ax=ax[i])
plt.show()


In [ ]:
for c in data.select_dtypes(object).columns:
    sns.pairplot(data, hue=c)
plt.show()

In [ ]:
data.columns

In [ ]:
data = pd.get_dummies(data, columns=['Company Type', 'WFH Setup Available',
       'Gender'], drop_first=True)
data.head()
encoded_columns = data.columns



In [ ]:
# Split df into X and y
y = data['Burn Rate']
X = data.drop('Burn Rate', axis=1)


In [ ]:
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.7, shuffle=True, random_state=1)

# Scale X
scaler = StandardScaler()
scaler.fit(X_train)
X_train = pd.DataFrame(scaler.transform(X_train), index=X_train.index, columns=X_train.columns)
X_test = pd.DataFrame(scaler.transform(X_test), index=X_test.index, columns=X_test.columns )

In [ ]:
import os
import pickle

scaler_filename = '../models/scaler.pkl'

# Create the directory if it doesn't exist
os.makedirs(os.path.dirname(scaler_filename), exist_ok=True)

# Save the scaler to the file
with open(scaler_filename, 'wb') as scaler_file:
    pickle.dump(scaler, scaler_file)

In [ ]:
X_train

In [ ]:
y_train

In [ ]:
import os

# Define the directory path
path = '../data/processed/'

# Create the directory if it does not exist
os.makedirs(path, exist_ok=True)

# Save the processed data
X_train.to_csv(path + 'X_train_processed.csv', index=False)
y_train.to_csv(path + 'y_train_processed.csv', index=False)


# **MODEL MAKING**

In [ ]:
from sklearn.linear_model import LinearRegression

# Create an instance of the LinearRegression class
linear_regression_model = LinearRegression()

# Train the model
linear_regression_model.fit(X_train, y_train)


In [ ]:
from sklearn.metrics import accuracy_score

# Make predictions
y_pred = linear_regression_model.predict(X_test)

# Convert regression output to 0/1 (because burnout is classification)
y_pred = (y_pred > 0.5).astype(int)

# Print accuracy
print("Accuracy:", accuracy_score(y_test, y_pred))

In [ ]:
pip install --upgrade scikit-learn


In [ ]:
#Linear Regressing Model Performance Metrics

print("Linear Regression Model Performance Metrics:\n")
# Make predictions on the test set
y_pred = linear_regression_model.predict(X_test)

# Calculate mean squared error
mse = mean_squared_error(y_test, y_pred)
print("Mean Squared Error:", mse)

# Calculate root mean squared error
rmse = np.sqrt(mse)
print("Root Mean Squared Error:", rmse)


# Calculate mean absolute error
mae = mean_absolute_error(y_test, y_pred)
print("Mean Absolute Error:", mae)

# Calculate R-squared score
r2 = r2_score(y_test, y_pred)
print("R-squared Score:", r2)

In [ ]:
feature_names = X.columns.tolist()
feature_names

In [ ]:
# Save the model to a file
model_filename = '../models/linear_regression.pkl'
with open (model_filename ,'wb') as  model_file:
    pickle.dump(linear_regression_model, model_file)

In [ ]:
# Create an instance of LinearSVR and explicitly set 'dual' to 'auto' to avoid the FutureWarning
SVMLinear = LinearSVR(dual='auto', max_iter=10000)  # You can adjust 'max_iter' as needed

# Fit the model
SVMLinear.fit(X_train, y_train)


In [ ]:
#Support Vector Machine (Linear Kernel) Performance Metrics
print("Support Vector Machine (Linear Kernel) Performance Metrics\n")
# Make predictions on the test set
y_pred = SVMLinear.predict(X_test)

# Calculate mean squared error
mse = mean_squared_error(y_test, y_pred)
print("Mean Squared Error:", mse)

# Calculate root mean squared error
rmse = np.sqrt(mse)
print("Root Mean Squared Error:", rmse)


# Calculate mean absolute error
mae = mean_absolute_error(y_test, y_pred)
print("Mean Absolute Error:", mae)

# Calculate R-squared score
r2 = r2_score(y_test, y_pred)
print("R-squared Score:", r2)


In [ ]:
SVMRbf = SVR()
SVMRbf.fit(X_train, y_train)

In [ ]:
#Support Vector Machine (RBF Kernel) Performance Metrics
print("Support Vector Machine (RBF Kernel) Performance Metrics\n")
# Make predictions on the test set
y_pred = SVMRbf.predict(X_test)

# Calculate mean squared error
mse = mean_squared_error(y_test, y_pred)
print("Mean Squared Error:", mse)

# Calculate root mean squared error
rmse = np.sqrt(mse)
print("Root Mean Squared Error:", rmse)

# Calculate mean absolute error
mae = mean_absolute_error(y_test, y_pred)
print("Mean Absolute Error:", mae)

# Calculate R-squared score
r2 = r2_score(y_test, y_pred)
print("R-squared Score:", r2)

In [ ]:
RandomForest = RandomForestRegressor()
RandomForest.fit(X_train, y_train)

In [ ]:
print("R-squared Score:", r2)
